In [1]:
# Retail360 - Silver Layer
# Step 1: Read Bronze Delta tables

customers = spark.read.table("Retail360_Bronze.dbo.customers")
products  = spark.read.table("Retail360_Bronze.dbo.products")
sales     = spark.read.table("Retail360_Bronze.dbo.sales")
stores    = spark.read.table("Retail360_Bronze.dbo.stores")

print("Bronze tables loaded successfully")

print("Customers:", customers.count())
print("Products :", products.count())
print("Sales    :", sales.count())
print("Stores   :", stores.count())

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 3, Finished, Available, Finished, False)

Bronze tables loaded successfully
Customers: 500
Products : 100
Sales    : 10000
Stores   : 20


In [2]:
# Retail360 - Silver Layer
# Step 2: Data Profiling

from pyspark.sql.functions import col, count, when, sum as spark_sum

tables = {
    "customers": customers,
    "products": products,
    "sales": sales,
    "stores": stores
}

for name, df in tables.items():
    print(f"\n===== {name.upper()} =====")
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")
    df.printSchema()

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 5, Finished, Available, Finished, False)


===== CUSTOMERS =====
Rows: 500
Columns: 5
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- signup_date: date (nullable = true)


===== PRODUCTS =====
Rows: 100
Columns: 4
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: double (nullable = true)


===== SALES =====
Rows: 10000
Columns: 12
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_am

In [3]:
# Sales data quality checks

print("===== SALES DATA QUALITY =====")

print("\nNull counts:")
sales.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in sales.columns
]).show()

print("\nDuplicate Order IDs:")
print(
    sales.groupBy("order_id")
         .count()
         .filter(col("count") > 1)
         .count()
)

print("\nInvalid Quantity:")
print(
    sales.filter(col("quantity") <= 0).count()
)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 6, Finished, Available, Finished, False)

===== SALES DATA QUALITY =====

Null counts:
+--------+----------+-----------+----------+--------+--------+----------+--------------+------------+------------+---------------+----------+
|order_id|order_date|customer_id|product_id|store_id|quantity|unit_price|payment_method|discount_pct|gross_amount|discount_amount|net_amount|
+--------+----------+-----------+----------+--------+--------+----------+--------------+------------+------------+---------------+----------+
|       0|         0|          2|         0|       0|       0|         0|             1|           0|           0|              0|         0|
+--------+----------+-----------+----------+--------+--------+----------+--------------+------------+------------+---------------+----------+


Duplicate Order IDs:
0

Invalid Quantity:
1


In [5]:
from pyspark.sql.functions import col, when, trim, upper, coalesce, lit

# ===== CLEAN SALES DATA =====

sales_clean = (
    sales
    # Remove records with missing critical IDs
    .filter(
        col("order_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("product_id").isNotNull() &
        col("store_id").isNotNull()
    )

    # Quantity must be greater than 0
    .filter(col("quantity") > 0)

    # Clean payment method
    .withColumn(
        "payment_method",
        upper(trim(col("payment_method")))
    )

    # Replace null payment method
    .withColumn(
        "payment_method",
        coalesce(col("payment_method"), lit("UNKNOWN"))
    )
)

print("===== CLEAN SALES =====")
print("Original rows :", sales.count())
print("Clean rows    :", sales_clean.count())
print("Removed rows  :", sales.count() - sales_clean.count())

sales_clean.show(5)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 9, Finished, Available, Finished, False)

===== CLEAN SALES =====
Original rows : 10000
Clean rows    : 9997
Removed rows  : 3
+--------+----------+-----------+----------+--------+--------+----------+--------------+------------+------------+---------------+----------+
|order_id|order_date|customer_id|product_id|store_id|quantity|unit_price|payment_method|discount_pct|gross_amount|discount_amount|net_amount|
+--------+----------+-----------+----------+--------+--------+----------+--------------+------------+------------+---------------+----------+
|O1000001|2025-03-01|    C100427|     P1007|    S108|       5|   8711.83|           UPI|           5|    43559.15|        2177.96|  41381.19|
|O1000002|2025-02-12|    C100497|     P1075|    S117|       1|    6797.2|   NET BANKING|          20|      6797.2|        1359.44|   5437.76|
|O1000003|2025-04-03|    C100316|     P1084|    S113|       3|   7361.69|   NET BANKING|          10|    22085.07|        2208.51|  19876.56|
|O1000004|2025-08-14|    C100486|     P1080|    S114|       1| 

In [6]:
# ===== CUSTOMERS DATA QUALITY =====

print("===== CUSTOMERS DATA QUALITY =====")

print("\nNull counts:")
customers.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in customers.columns
]).show()

print("\nDuplicate Customer IDs:")
print(
    customers.groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("\nCustomers with blank names:")
print(
    customers.filter(
        col("customer_name").isNull() |
        (trim(col("customer_name")) == "")
    ).count()
)

print("\nCustomers with blank cities:")
print(
    customers.filter(
        col("city").isNull() |
        (trim(col("city")) == "")
    ).count()
)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 11, Finished, Available, Finished, False)

===== CUSTOMERS DATA QUALITY =====

Null counts:
+-----------+-------------+----+-------+-----------+
|customer_id|customer_name|city|segment|signup_date|
+-----------+-------------+----+-------+-----------+
|          0|            0|   0|      0|          0|
+-----------+-------------+----+-------+-----------+


Duplicate Customer IDs:
0

Customers with blank names:
0

Customers with blank cities:
0


In [7]:
print("===== PRODUCTS DATA QUALITY =====")

print("\nNull counts:")
products.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in products.columns
]).show()

print("\nDuplicate Product IDs:")
print(
    products.groupBy("product_id")
            .count()
            .filter(col("count") > 1)
            .count()
)

print("\nBlank Product Names:")
print(
    products.filter(
        col("product_name").isNull() |
        (trim(col("product_name")) == "")
    ).count()
)

print("\nBlank Categories:")
print(
    products.filter(
        col("category").isNull() |
        (trim(col("category")) == "")
    ).count()
)

print("\nInvalid Unit Prices:")
print(
    products.filter(
        col("unit_price").isNull() |
        (col("unit_price") <= 0)
    ).count()
)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 12, Finished, Available, Finished, False)

===== PRODUCTS DATA QUALITY =====

Null counts:
+----------+------------+--------+----------+
|product_id|product_name|category|unit_price|
+----------+------------+--------+----------+
|         0|           0|       0|         0|
+----------+------------+--------+----------+


Duplicate Product IDs:
0

Blank Product Names:
0

Blank Categories:
0

Invalid Unit Prices:
0


In [8]:
print("===== STORES DATA QUALITY =====")

print("\nNull counts:")
stores.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in stores.columns
]).show()

print("\nDuplicate Store IDs:")
print(
    stores.groupBy("store_id")
          .count()
          .filter(col("count") > 1)
          .count()
)
print("\nBlank Store Names:")
print(
    stores.filter(
        col("store_name").isNull() |
        (trim(col("store_name")) == "")
    ).count()
)

print("\nBlank Cities:")
print(
    stores.filter(
        col("city").isNull() |
        (trim(col("city")) == "")
    ).count()
)

print("\nBlank Store Types:")
print(
    stores.filter(
        col("store_type").isNull() |
        (trim(col("store_type")) == "")
    ).count()
)


StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 13, Finished, Available, Finished, False)

===== STORES DATA QUALITY =====

Null counts:
+--------+----------+----+----------+
|store_id|store_name|city|store_type|
+--------+----------+----+----------+
|       0|         0|   0|         0|
+--------+----------+----+----------+


Duplicate Store IDs:
0

Blank Store Names:
0

Blank Cities:
0

Blank Store Types:
0


In [9]:
# ===== CLEAN PRODUCTS =====

products_clean = (
    products
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .filter(col("product_id").isNotNull())
    .filter(col("product_name").isNotNull())
    .filter(col("unit_price") > 0)
    .dropDuplicates(["product_id"])
)

print("Original Products:", products.count())
print("Clean Products:", products_clean.count())

products_clean.show(5)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 14, Finished, Available, Finished, False)

Original Products: 100
Clean Products: 100
+----------+------------+-----------+----------+
|product_id|product_name|   category|unit_price|
+----------+------------+-----------+----------+
|     P1001|   Product_1|    Grocery|  17228.04|
|     P1002|   Product_2|     Beauty|   1788.21|
|     P1003|   Product_3|     Beauty|   7593.29|
|     P1004|   Product_4|Electronics|  17733.19|
|     P1005|   Product_5|Electronics|    1776.1|
+----------+------------+-----------+----------+
only showing top 5 rows



In [10]:
# ===== CLEAN STORES =====

stores_clean = (
    stores
    .withColumn("store_name", trim(col("store_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("store_type", trim(col("store_type")))
    .filter(col("store_id").isNotNull())
    .filter(col("store_name").isNotNull())
    .filter(col("city").isNotNull())
    .dropDuplicates(["store_id"])
)

print("Original Stores:", stores.count())
print("Clean Stores:", stores_clean.count())

stores_clean.show(5)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 16, Finished, Available, Finished, False)

Original Stores: 20
Clean Stores: 20
+--------+-----------------+---------+----------+
|store_id|       store_name|     city|store_type|
+--------+-----------------+---------+----------+
|    S101|Retail360 Store 1|   Mumbai|    Outlet|
|    S102|Retail360 Store 2|   Mumbai|      Mall|
|    S103|Retail360 Store 3|     Pune|    Outlet|
|    S104|Retail360 Store 4|Bengaluru|      Mall|
|    S105|Retail360 Store 5|   Mumbai|    Outlet|
+--------+-----------------+---------+----------+
only showing top 5 rows



In [11]:
# ===== CLEAN CUSTOMERS =====

customers_clean = (
    customers
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("segment", trim(col("segment")))
    .filter(col("customer_id").isNotNull())
    .filter(col("customer_name").isNotNull())
    .filter(col("city").isNotNull())
    .dropDuplicates(["customer_id"])
)

print("Original Customers:", customers.count())
print("Clean Customers:", customers_clean.count())

customers_clean.show(5)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 18, Finished, Available, Finished, False)

Original Customers: 500
Clean Customers: 500
+-----------+-------------+---------+--------------+-----------+
|customer_id|customer_name|     city|       segment|signup_date|
+-----------+-------------+---------+--------------+-----------+
|    C100001|   Customer_1|Hyderabad|      Consumer| 2022-07-17|
|    C100002|   Customer_2|Ahmedabad|      Consumer| 2023-08-02|
|    C100003|   Customer_3|     Pune|Small Business| 2023-01-26|
|    C100004|   Customer_4|Hyderabad|      Consumer| 2025-04-17|
|    C100005|   Customer_5|   Jaipur|      Consumer| 2023-07-08|
+-----------+-------------+---------+--------------+-----------+
only showing top 5 rows



In [12]:
from pyspark.sql.functions import (
    col, upper, trim, coalesce, lit
)
import pyspark.sql.functions as F

# ===== STANDARDIZE SALES =====

sales_silver = (
    sales_clean
    .withColumn(
        "payment_method",
        upper(trim(col("payment_method")))
    )
    .withColumn(
        "payment_method",
        coalesce(col("payment_method"), lit("UNKNOWN"))
    )
    .withColumn(
        "gross_amount",
        F.round(col("gross_amount"), 2)
    )
    .withColumn(
        "discount_amount",
        F.round(col("discount_amount"), 2)
    )
    .withColumn(
        "net_amount",
        F.round(col("net_amount"), 2)
    )
)

print("Silver Sales Rows:", sales_silver.count())

sales_silver.show(5)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 20, Finished, Available, Finished, False)

Silver Sales Rows: 9997
+--------+----------+-----------+----------+--------+--------+----------+--------------+------------+------------+---------------+----------+
|order_id|order_date|customer_id|product_id|store_id|quantity|unit_price|payment_method|discount_pct|gross_amount|discount_amount|net_amount|
+--------+----------+-----------+----------+--------+--------+----------+--------------+------------+------------+---------------+----------+
|O1000001|2025-03-01|    C100427|     P1007|    S108|       5|   8711.83|           UPI|           5|    43559.15|        2177.96|  41381.19|
|O1000002|2025-02-12|    C100497|     P1075|    S117|       1|    6797.2|   NET BANKING|          20|      6797.2|        1359.44|   5437.76|
|O1000003|2025-04-03|    C100316|     P1084|    S113|       3|   7361.69|   NET BANKING|          10|    22085.07|        2208.51|  19876.56|
|O1000004|2025-08-14|    C100486|     P1080|    S114|       1|    1903.0|   NET BANKING|           5|      1903.0|          

In [13]:
# ===== STANDARDIZE CUSTOMERS =====

customers_silver = (
    customers
    .withColumn("customer_name", F.trim(col("customer_name")))
    .withColumn("city", F.upper(F.trim(col("city"))))
    .withColumn("segment", F.upper(F.trim(col("segment"))))
)

print("Silver Customers Rows:", customers_silver.count())

customers_silver.show(5)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 22, Finished, Available, Finished, False)

Silver Customers Rows: 500
+-----------+-------------+---------+--------------+-----------+
|customer_id|customer_name|     city|       segment|signup_date|
+-----------+-------------+---------+--------------+-----------+
|    C100001|   Customer_1|HYDERABAD|      CONSUMER| 2022-07-17|
|    C100002|   Customer_2|AHMEDABAD|      CONSUMER| 2023-08-02|
|    C100003|   Customer_3|     PUNE|SMALL BUSINESS| 2023-01-26|
|    C100004|   Customer_4|HYDERABAD|      CONSUMER| 2025-04-17|
|    C100005|   Customer_5|   JAIPUR|      CONSUMER| 2023-07-08|
+-----------+-------------+---------+--------------+-----------+
only showing top 5 rows



In [14]:
# ===== STANDARDIZE PRODUCTS =====

products_silver = (
    products
    .withColumn("product_name", F.trim(col("product_name")))
    .withColumn("category", F.upper(F.trim(col("category"))))
    .withColumn("unit_price", F.round(col("unit_price"), 2))
)

print("Silver Products Rows:", products_silver.count())

products_silver.show(5)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 24, Finished, Available, Finished, False)

Silver Products Rows: 100
+----------+------------+-----------+----------+
|product_id|product_name|   category|unit_price|
+----------+------------+-----------+----------+
|     P1001|   Product_1|    GROCERY|  17228.04|
|     P1002|   Product_2|     BEAUTY|   1788.21|
|     P1003|   Product_3|     BEAUTY|   7593.29|
|     P1004|   Product_4|ELECTRONICS|  17733.19|
|     P1005|   Product_5|ELECTRONICS|    1776.1|
+----------+------------+-----------+----------+
only showing top 5 rows



In [15]:
# ===== STANDARDIZE STORES =====

stores_silver = (
    stores
    .withColumn("store_name", F.trim(col("store_name")))
    .withColumn("city", F.upper(F.trim(col("city"))))
    .withColumn("store_type", F.upper(F.trim(col("store_type"))))
)

print("Silver Stores Rows:", stores_silver.count())

stores_silver.show(5)

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 26, Finished, Available, Finished, False)

Silver Stores Rows: 20
+--------+-----------------+---------+----------+
|store_id|       store_name|     city|store_type|
+--------+-----------------+---------+----------+
|    S101|Retail360 Store 1|   MUMBAI|    OUTLET|
|    S102|Retail360 Store 2|   MUMBAI|      MALL|
|    S103|Retail360 Store 3|     PUNE|    OUTLET|
|    S104|Retail360 Store 4|BENGALURU|      MALL|
|    S105|Retail360 Store 5|   MUMBAI|    OUTLET|
+--------+-----------------+---------+----------+
only showing top 5 rows



In [16]:
print("===== SALES SCHEMA =====")
sales_silver.printSchema()

print("===== CUSTOMERS SCHEMA =====")
customers_silver.printSchema()

print("===== PRODUCTS SCHEMA =====")
products_silver.printSchema()

print("===== STORES SCHEMA =====")
stores_silver.printSchema()

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 28, Finished, Available, Finished, False)

===== SALES SCHEMA =====
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- discount_pct: integer (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)

===== CUSTOMERS SCHEMA =====
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- signup_date: date (nullable = true)

===== PRODUCTS SCHEMA =====
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: double (nullable = true)

===== STORES SC

In [17]:
# ===== SAVE SILVER TABLES =====

sales_silver.write.mode("overwrite").format("delta").saveAsTable("sales")

customers_silver.write.mode("overwrite").format("delta").saveAsTable("customers")

products_silver.write.mode("overwrite").format("delta").saveAsTable("products")

stores_silver.write.mode("overwrite").format("delta").saveAsTable("stores")

print("Silver tables saved successfully!")

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 29, Finished, Available, Finished, False)

Silver tables saved successfully!


In [18]:
# ===== VERIFY SILVER TABLES =====

print("Sales:", spark.table("sales").count())
print("Customers:", spark.table("customers").count())
print("Products:", spark.table("products").count())
print("Stores:", spark.table("stores").count())

StatementMeta(, 20d6d2d6-9ad4-48ca-a38c-bda35afcf50c, 30, Finished, Available, Finished, False)

Sales: 9997
Customers: 500
Products: 100
Stores: 20
